# FHOPS tactical–operational planning walkthrough

This guided notebook demonstrates the Phase 6/7 TOPM-inspired workflow: generate a redistributable practitioner-scale scenario, validate the contract, solve the aggregate MILP, audit product/inventory balances, compare an overlay, and compile selected commitments into an operational business window.

The case is synthetic and redistributable. It is not a calibrated empirical forecast.

In [ ]:
import json
import tempfile
from pathlib import Path

import pandas as pd

from fhops.model.milp.tactical_operational import (
    build_tactical_operational_bundle,
    solve_tactical_operational_milp,
)
from fhops.planning.tactical_operational.integration import (
    commitments_from_result,
    compile_business_window_scenario,
    write_operational_scenario_bundle,
)
from fhops.planning.tactical_operational.io import load_tactical_operational_scenario
from fhops.planning.tactical_operational.practitioner import generate_tactical_practitioner_case
from fhops.planning.tactical_operational.scenario import (
    diff_tactical_scenarios,
    load_tactical_overlay_scenario,
    write_tactical_report,
    write_tactical_scenario_yaml,
)
from fhops.scenario.io import load_scenario

workdir = Path(tempfile.mkdtemp(prefix="fhops_tactical_notebook_"))
print("Working directory:", workdir)

## 1. Generate and validate the tactical contract

The practitioner case includes 24 blocks, 8 periods, 3 products, 2 facilities, 3 harvest systems, roads, silviculture transitions, external supply, and an optional fleet investment.

In [ ]:
scenario = generate_tactical_practitioner_case()
scenario_path = workdir / "practitioner.yaml"
write_tactical_scenario_yaml(scenario, scenario_path)
loaded = load_tactical_operational_scenario(scenario_path)

dimensions = loaded.dimension_summary()
pd.Series(dimensions).to_frame("count")

## 2. Solve the aggregate tactical MILP

We enable roads, silviculture, and fleet investment so the result exercises the full Phase 6 module stack.

In [ ]:
bundle = build_tactical_operational_bundle(
    loaded,
    enable_roads=True,
    enable_silviculture=True,
    enable_fleet_investment=True,
)
result = solve_tactical_operational_milp(
    bundle,
    solver="highs",
    time_limit=120,
    gap=0.01,
)
print("status:", result["solver_status"], "| termination:", result["termination_condition"])
print("objective:", round(result["objective"], 3))
pd.Series(result["objective_components"]).to_frame("value")

## 3. Inspect decision tables

Harvest decisions are area-based; product production, flows, inventory, roads, silviculture, and fleet tables are normalized for downstream reporting.

In [ ]:
harvest = result["harvest_decisions"]
flows = result["flows"]
inventory = result["inventory"]
roads = result["roads"]
fleet = result["fleet"]

print("active harvest options:", len(harvest))
print("nonzero flows:", len(flows))
print("active road rows:", len(roads))
print("fleet purchases:", len(fleet))
harvest[["block_id", "system_id", "period_id", "harvested_area_ha", "total_volume_m3"]].head(10)

## 4. Audit inventory balances

Every facility/product/period should satisfy opening + deliveries + purchases − consumption = closing.

In [ ]:
opening_lookup = {
    (row.facility_id, row.product_id): row.opening_m3 for row in loaded.initial_inventory
}
closing_lookup = {
    (row.facility_id, row.product_id, row.period_id): float(row.closing_m3)
    for row in inventory.itertuples(index=False)
}
periods = sorted(loaded.periods, key=lambda period: period.sequence)
errors = []
for facility in loaded.facilities:
    for product_id in facility.accepted_products:
        opening = opening_lookup.get((facility.facility_id, product_id), 0.0)
        for period in periods:
            key = (facility.facility_id, product_id, period.period_id)
            delivered = flows.loc[
                (flows["destination_id"] == facility.facility_id)
                & (flows["product_id"] == product_id)
                & (flows["period_id"] == period.period_id),
                "volume_m3",
            ].sum()
            purchased = (
                result["purchases"]
                .loc[
                    (result["purchases"]["destination_id"] == facility.facility_id)
                    & (result["purchases"]["product_id"] == product_id)
                    & (result["purchases"]["period_id"] == period.period_id),
                    "volume_m3",
                ]
                .sum()
                if not result["purchases"].empty
                else 0.0
            )
            consumed = (
                result["consumption"]
                .loc[
                    (result["consumption"]["facility_id"] == facility.facility_id)
                    & (result["consumption"]["product_id"] == product_id)
                    & (result["consumption"]["period_id"] == period.period_id),
                    "consumption_m3",
                ]
                .sum()
            )
            expected = opening + delivered + purchased - consumed
            closing = closing_lookup.get(key, 0.0)
            if abs(expected - closing) > 1e-5:
                errors.append((key, expected, closing))
            opening = closing
assert not errors, errors[:5]
print("Inventory balances verified for all facility/product/period keys.")

## 5. Compare a sparse overlay

Overlays let us change a few rows without copying the full scenario. Here we reduce first-period sawmill demand and diff the result against the base contract.

In [ ]:
overlay_path = workdir / "lower-demand.yaml"
overlay_path.write_text("""overlay_id: lower-demand
facility_demand:
  - facility_id: coastal_sawmill
    product_id: sawlog
    period_id: Y2027-P01
    target_m3: 2000.0
""")
overlaid = load_tactical_overlay_scenario(scenario_path, overlay_path)
diff = diff_tactical_scenarios(loaded, overlaid)
print("changed fields:", len(diff))
diff[["section", "field", "base_value", "candidate_value"]].head()

## 6. Compile a tactical commitment window into an operational scenario

The handoff uses an existing operational bundle for machines/calendars/rates. We map the first two selected tactical blocks onto tiny7's B01/B02 operational blocks and write a loadable operational scenario bundle.

In [ ]:
commitments = commitments_from_result(result)
block_map = {}
for index, commitment in enumerate(commitments[:2], start=1):
    block_map[commitment.block_id] = f"B{index:02d}"

examples_dir = Path.cwd() if (Path.cwd() / "tiny7").is_dir() else Path.cwd() / "examples"
operational_base = load_scenario(examples_dir / "tiny7" / "scenario.yaml")
compiled = compile_business_window_scenario(
    operational_base,
    commitments[:2],
    block_map=block_map,
    start_day=1,
    horizon_days=7,
)
compiled_path = write_operational_scenario_bundle(compiled, workdir / "operational-window")
compiled_loaded = load_scenario(compiled_path)
print("compiled blocks:", [block.id for block in compiled_loaded.blocks])
print("systems:", [block.harvest_system_id for block in compiled_loaded.blocks])

## 7. Write normalized reports

Reports can be written as CSV/Parquet/Markdown and shared without rerunning the solver.

In [ ]:
result_json = workdir / "result.json"
serializable = dict(result)
for key, value in list(serializable.items()):
    if isinstance(value, pd.DataFrame):
        serializable[key] = value.to_dict("records")
result_json.write_text(json.dumps(serializable, indent=2), encoding="utf-8")
written = write_tactical_report(result, workdir / "report", formats="csv,markdown")
print("wrote", len(written), "report artifacts")
(sorted(path.name for path in written.values()))

## Next steps

- Use `fhops scenario batch` to run overlay ensembles.
- Use `fhops scenario benchmark` to profile model size/runtime before considering decomposition.
- Use `fhops plan compile-tactical` when you want the same handoff from the CLI.
- See `docs/howto/tactical_operational.rst` for the canonical formulation and command reference.